# 00 — Business Understanding

| | |
|---|---|
| **CRISP-DM Phase** | Phase 1 — Business Understanding |
| **Goal** | Define the business problem, churn label, success criteria, and pipeline map |
| **Inputs** | None |
| **Outputs** | Context consumed by all downstream notebooks — no data artifact |

---
## 1. Olist — Business Context

Olist is a Brazilian e-commerce marketplace that connects small-to-medium sellers to major retail channels (Mercado Livre, Americanas, etc.) under a single platform. Revenue comes from seller subscriptions and a commission on each completed sale — making **Gross Merchandise Value (GMV) the primary revenue lever**.

Because repeat buyers drive GMV without additional acquisition cost, **customer retention is a direct revenue driver**, not a secondary CX concern.

---
## 2. Problem Statement

The dataset shows that the vast majority of Olist customers (~97%) make only a single purchase and never return. Olist currently has no systematic way to distinguish:

- Customers who **will return** without any intervention
- Customers who are **at risk** but recoverable with a targeted offer
- Customers who are **permanently lost**

Without this signal, retention spend is either wasted on customers who would have returned anyway, or absent for customers who needed outreach.

**Objective:** Build a binary classifier that scores each customer's churn risk after their most recent order, enabling the retention team to prioritise outreach on high-risk customers.

---
## 3. Churn Definition

> **A customer is churned (label = 1) if they make no repeat purchase within 90 days of their last completed order.**

### Why 90 days?

| Window | Verdict |
|---|---|
| 30 days | Too short — most e-commerce categories have natural repurchase cycles > 30 days. Over-labels customers as churned. |
| 60 days | Borderline — still undershoots for durable goods common on Olist. |
| **90 days** ✓ | Industry standard for e-commerce churn. Captures two full monthly cycles. Long enough to observe genuine inactivity; short enough for actionable intervention. |
| 120 days | Too long — by the time the window closes, the customer has likely transacted with a competitor and recovery is unlikely. |

### Label construction logic

```
1. dataset_end_date  = max(order_purchase_timestamp)   # delivered orders only
2. cutoff_date       = dataset_end_date − 90 days
3. eligible          = customers where last_order_date ≤ cutoff_date
   # customers after the cutoff cannot be labelled — the 90-day window is not yet observable
4. churn = 1  if no order in (last_order_date, last_order_date + 90 days]
   churn = 0  if at least one order exists in that window
```

The cutoff exclusion in step 3 is critical — including customers whose 90-day window extends beyond the dataset end would introduce systematic label noise.

---
## 4. Success Criteria

| Metric | Target | Rationale |
|---|---|---|
| **ROC-AUC** | ≥ 0.75 | Primary. Threshold-independent; robust to class imbalance. |
| **PR-AUC** | Maximise | Secondary. At ~97/3 imbalance, more discriminating than ROC-AUC for minority-class performance. |

> **Accuracy is excluded.** A model that predicts everyone churns achieves ~97% accuracy — it is useless for targeting. Accuracy will not appear as a decision criterion anywhere in this project.

---
## 5. Dataset Overview

**Source:** Brazilian E-Commerce Public Dataset by Olist (`olistbr/brazilian-ecommerce` on Kaggle)  
**Period:** October 2016 – October 2018  
**Download:** `kagglehub` only — no manual downloads

| # | Table | Key Contents |
|---|---|---|
| 1 | `olist_orders_dataset` | Order lifecycle: status, all timestamps |
| 2 | `olist_order_items_dataset` | Line items: product, seller, price, freight |
| 3 | `olist_order_payments_dataset` | Payment method, installments, value — **multiple rows per order** |
| 4 | `olist_order_reviews_dataset` | Review score (1–5), text, timestamps |
| 5 | `olist_customers_dataset` | Customer ID mapping and zip code |
| 6 | `olist_sellers_dataset` | Seller ID and zip code |
| 7 | `olist_products_dataset` | Category, weight, dimensions |
| 8 | `olist_geolocation_dataset` | Zip → lat/lon/city/state — **~1M rows** |
| 9 | `product_category_name_translation` | Portuguese category names → English |

### Known structural issues — flagged upfront

| Issue | Impact | Handled In |
|---|---|---|
| `customer_id` is order-scoped; `customer_unique_id` is the true customer identifier | All customer-level analysis must use `customer_unique_id` | `01`, `02` |
| Payments table has multiple rows per order | Must aggregate to order level before joining | `02` |
| Geolocation table has ~1M rows with multiple entries per zip | Join at state level only | `02` |
| Not all orders have reviews | Add `has_review` binary flag; do not drop | `02`, `04` |
| Product categories in Portuguese | Apply translation table before any category analysis | `02` |

---
## 6. Key Assumptions and Risks

| Item | Detail |
|---|---|
| **Delivered orders only** | Only `order_status = 'delivered'` orders are used. Cancelled and in-transit orders are excluded. |
| **Class imbalance (~97/3)** | Treated as a first-class modelling challenge. Drives metric selection, model choice, and threshold tuning. |
| **No random train/test split** | Temporal data requires a time-based split to prevent data leakage. |
| **No NLP features** | Review comment text is not used in this iteration. Flagged as a future enhancement. |
| **No customer demographics** | Model relies entirely on transactional signals. |
| **Geographic bias** | São Paulo dominates order volume. Model performance may be weaker in low-volume regions. |

---
## 7. CRISP-DM Pipeline Map

```
┌─────────────────────────────────────────────────────────────────────┐
│  Phase 1 — Business Understanding                                   │
│  📓 00_business_understanding.ipynb   [this notebook]               │
│  Output: context only                                               │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 2 — Data Understanding                                       │
│  📓 01_data_understanding.ipynb                                     │
│  Output: 01_profiling_summary.parquet                               │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 3 — Data Preparation                                         │
│  📓 02_data_preparation.ipynb                                       │
│  Output: 02_master_table.parquet                                    │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 2 (extended) — Exploratory Data Analysis                     │
│  📓 03_eda.ipynb                                                    │
│  Output: 03_eda_outputs.parquet                                     │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 3 (extended) — Feature Engineering                           │
│  📓 04_feature_engineering.ipynb                                    │
│  Output: 04_features.parquet                                        │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 4 — Modelling                                                │
│  📓 05_modeling.ipynb                                               │
│  Output: 05_best_model.joblib                                       │
│          05_predictions.parquet                                     │
│          05_model_comparison.parquet                                │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 5 — Evaluation                                               │
│  📓 06_evaluation.ipynb                                             │
│  Output: 06_shap_values.parquet                                     │
│          06_risk_segments.parquet                                   │
└───────────────────────────┬─────────────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────────────┐
│  Phase 6 — Communication                                            │
│  📓 07_summary.ipynb                                                │
│  Output: portfolio deliverable                                      │
└─────────────────────────────────────────────────────────────────────┘
```